# Proyecto Final: Optimizacion de Redes Neuronales con Algoritmos Geneticos

Libreta resumen del proyecto. Se desglosa el trabajo modularizado en pasos ejecutables para documentacion formal.

### Objetivo
Encontrar una arquitectura de Red Neuronal Convolucional (CNN) que:
1. Tenga la mayor exactitud posible (val_accuracy).
2. Tenga la menor cantidad de parametros (pesos) posible (trainable_params).


In [ ]:
# Importamos las librerias necesarias
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.datasets import cifar10
import pygad
import matplotlib.pyplot as plt
import time
import os
import warnings

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
warnings.filterwarnings('ignore')


## 1. Configuracion e Hiperparametros

Se definen los limites y constantes del experimento.

- Epoch: Ciclo completo sobre el dataset de entrenamiento.
- Batch Size: Tamano de lote procesado antes de actualizar los pesos.
- Learning Rate: Tasa de aprendizaje del optimizador.
- TP (Parametros entrenables): Cantidad de pesos en la red.


In [ ]:
GA_TAMANIO_POBLACION = 12     
GA_NUM_GENERACIONES = 12      
GA_EPOCAS_INDIVIDUO = 5       

FITNESS_W1 = 0.7              
FITNESS_W2 = 0.3              
TP_MIN = 10000               
TP_MAX = 1000000              

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)


## 2. Preparacion de los Datos (CIFAR-10)

Division del conjunto de datos:
- Train (60%): Entrenamiento de la red.
- Validation (20%): Evaluacion del Algoritmo Genetico.
- Test (20%): Evaluacion final.


In [ ]:
def cargar_cifar10():
    print("[INFO] Descargando y preparando CIFAR-10...")
    (x_full, y_full), (x_test, y_test) = cifar10.load_data()
    
    x_all = np.concatenate((x_full, x_test), axis=0)
    y_all = np.concatenate((y_full, y_test), axis=0)
    
    x_all = x_all.astype("float32") / 255.0
    
    indices = np.arange(len(x_all))
    np.random.shuffle(indices)
    x_all = x_all[indices]
    y_all = y_all[indices]
    
    n_train = int(0.6 * len(x_all))
    n_val   = int(0.2 * len(x_all))
    
    x_train, y_train = x_all[:n_train], y_all[:n_train]
    x_val, y_val     = x_all[n_train:n_train+n_val], y_all[n_train:n_train+n_val]
    x_test, y_test   = x_all[n_train+n_val:], y_all[n_train+n_val:]
    
    print(f"Datos listos. Train: {len(x_train)}, Val: {len(x_val)}, Test: {len(x_test)}")
    return (x_train, y_train), (x_val, y_val), (x_test, y_test)

(x_train, y_train), (x_val, y_val), (x_test, y_test) = cargar_cifar10()


## 3. Fase 1: Arquitectura Base (Baseline)

Modelo de referencia construido segun la estructura del proyecto.


In [ ]:
def construir_red_base():
    modelo = Sequential(name="Red_Base")
    
    modelo.add(Conv2D(32, (3,3), activation="relu", input_shape=(32,32,3)))
    modelo.add(MaxPooling2D((2,2)))
    
    modelo.add(Conv2D(64, (3,3), activation="relu"))
    modelo.add(MaxPooling2D((2,2)))
    
    modelo.add(Flatten())
    modelo.add(Dense(128, activation="relu"))
    modelo.add(Dense(10, activation="softmax"))
    
    modelo.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return modelo

print("Entrenando modelo Base (30 epocas segun rubrica)...")
modelo_base = construir_red_base()
historial_base = modelo_base.fit(x_train, y_train, epochs=30, validation_data=(x_val, y_val), batch_size=64, verbose=1)
acc_test_base = modelo_base.evaluate(x_test, y_test, verbose=0)[1]
print(f"Accuracy en Test (Baseline): {acc_test_base*100:.2f}% con {modelo_base.count_params()} parametros.")


## 4. Fase 2: Codificacion Genetica

Traduccion de la arquitectura a un vector (cromosoma) evaluable por pyGAD.


In [ ]:
ESPACIO_GENES = [
    [0.01, 0.005, 0.001, 0.0005, 0.0001], # Gen 0: Learning Rate
    [32, 64, 128],                        # Gen 1: Batch Size
    [0, 1, 2],                            # Gen 2: Optimizador (0:Adam, 1:RMSprop, 2:SGD)
    [1, 2, 3],                            # Gen 3: Num capas Conv2D
    [16, 32, 64, 128, 256],               # Gen 4: Filtros Capa 1
    [16, 32, 64, 128, 256],               # Gen 5: Filtros Capa 2
    [16, 32, 64, 128, 256],               # Gen 6: Filtros Capa 3
    [3, 5],                               # Gen 7: Tamano del Kernel
    [64, 128, 256, 512],                  # Gen 8: Neuronas Capa Densa
    [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]        # Gen 9: Dropout
]

def decodificar_cromosoma(cromosoma):
    return {
        "lr": ESPACIO_GENES[0][int(cromosoma[0])],
        "batch": ESPACIO_GENES[1][int(cromosoma[1])],
        "opt_idx": ESPACIO_GENES[2][int(cromosoma[2])],
        "n_capas": ESPACIO_GENES[3][int(cromosoma[3])],
        "filtros": [
            ESPACIO_GENES[4][int(cromosoma[4])],
            ESPACIO_GENES[5][int(cromosoma[5])],
            ESPACIO_GENES[6][int(cromosoma[6])]
        ],
        "kernel": ESPACIO_GENES[7][int(cromosoma[7])],
        "dense": ESPACIO_GENES[8][int(cromosoma[8])],
        "dropout": ESPACIO_GENES[9][int(cromosoma[9])]
    }


## 5. Fase 3: Funcion Fitness

Optimizacion multiobjetivo para maximizar accuracy y minimizar parametros.


In [ ]:
def construir_red_genetica(params):
    modelo = Sequential()
    k_size = (params["kernel"], params["kernel"])
    
    dim_actual = 32
    capas_validas = 0
    for i in range(params["n_capas"]):
        dim_actual = dim_actual - params["kernel"] + 1 
        dim_actual = dim_actual // 2                   
        if dim_actual > 0:
            capas_validas += 1
        else:
            break
            
    for i in range(capas_validas):
        if i == 0:
            modelo.add(Conv2D(params["filtros"][i], k_size, activation="relu", input_shape=(32,32,3)))
        else:
            modelo.add(Conv2D(params["filtros"][i], k_size, activation="relu"))
        modelo.add(MaxPooling2D((2,2)))
        
    modelo.add(Flatten())
    if params["dropout"] > 0:
        modelo.add(Dropout(params["dropout"]))
    modelo.add(Dense(params["dense"], activation="relu"))
    modelo.add(Dense(10, activation="softmax"))
    
    opt = tf.keras.optimizers.Adam(params["lr"])
    if params["opt_idx"] == 1: opt = tf.keras.optimizers.RMSprop(params["lr"])
    elif params["opt_idx"] == 2: opt = tf.keras.optimizers.SGD(params["lr"])
        
    modelo.compile(optimizer=opt, loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return modelo

def fitness_func(ga_instance, solucion, solucion_idx):
    params = decodificar_cromosoma(solucion)
    modelo = construir_red_genetica(params)
    tp = modelo.count_params()
    
    hist = modelo.fit(x_train, y_train, epochs=GA_EPOCAS_INDIVIDUO, validation_data=(x_val, y_val), batch_size=params["batch"], verbose=0)
    val_acc = hist.history["val_accuracy"][-1]
    
    tp_penalizado = np.clip((tp - TP_MIN) / (TP_MAX - TP_MIN), 0, 1)
    fitness = FITNESS_W1 * val_acc + FITNESS_W2 * (1 - tp_penalizado)
    
    print(f"Evaluacion -> Acc: {val_acc:.4f} | Parametros: {tp} | Fitness: {fitness:.4f}")
    
    tf.keras.backend.clear_session()
    return fitness


## 6. Fase 4: Ejecucion del Algoritmo Genetico

Ejecucion de pyGAD con la poblacion y generaciones definidas.


In [ ]:
ga_instance = pygad.GA(
    num_generations=GA_NUM_GENERACIONES,
    num_parents_mating=6,
    fitness_func=fitness_func,
    sol_per_pop=GA_TAMANIO_POBLACION,
    num_genes=len(ESPACIO_GENES),
    gene_space=[list(range(len(esp))) for esp in ESPACIO_GENES],
    mutation_probability=0.25,
    crossover_type="single_point",
    mutation_type="random",
    keep_elitism=2
)

print("Iniciando optimizacion...")
ga_instance.run()

solucion, solucion_fitness, _ = ga_instance.best_solution()
mejores_params = decodificar_cromosoma(solucion)
print("Optimizacion terminada.")
print(f"Mejor Fitness: {solucion_fitness:.4f}")
print(f"Configuracion optima: {mejores_params}")


## 7. Fase 5: Reentrenamiento de la Arquitectura Optimizada

Entrenamiento intensivo (30 epocas) del mejor individuo para comparacion final.


In [ ]:
print("Reentrenando modelo optimizado desde cero (30 epocas)...")

modelo_campeon = construir_red_genetica(mejores_params)
historial_campeon = modelo_campeon.fit(
    x_train, y_train, 
    epochs=30, 
    validation_data=(x_val, y_val), 
    batch_size=mejores_params["batch"], 
    verbose=1
)

acc_test_campeon = modelo_campeon.evaluate(x_test, y_test, verbose=0)[1]
print(f"Accuracy en Test (Optimizado): {acc_test_campeon*100:.2f}% con {modelo_campeon.count_params()} parametros.")


## 8. Resultados y Comparacion Final

Visualizacion de metricas y comparacion contra el baseline.


In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(historial_base.history['val_loss'], label='Baseline', linestyle='--')
plt.plot(historial_campeon.history['val_loss'], label='Optimizado (GA)', linewidth=2)
plt.title('Perdida (Loss) en Validacion')
plt.xlabel('Epoca')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(historial_base.history['val_accuracy'], label='Baseline', linestyle='--')
plt.plot(historial_campeon.history['val_accuracy'], label='Optimizado (GA)', linewidth=2)
plt.title('Exactitud (Accuracy) en Validacion')
plt.xlabel('Epoca')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

print("="*50)
print(f"{'Metrica':<20} | {'Baseline':<10} | {'Optimizado (GA)':<10}")
print("-" * 50)
print(f"{'Accuracy en Test':<20} | {acc_test_base*100:8.2f}% | {acc_test_campeon*100:8.2f}%")
print(f"{'Parametros (TP)':<20} | {modelo_base.count_params():<10} | {modelo_campeon.count_params():<10}")
print("="*50)
